# LOREVERSE Gemini 비교 점검

아래 셀 왼쪽 ▶을 누르고 InfinityFree에 설정한 모델 ID와 API 키를 입력하세요.
키는 숨김 입력으로 받아 Google Gemini API에 HTTPS 헤더로 전송합니다.
셀 아래에는 HTTP 상태와 짧은 문장 응답 검증 결과만 표시합니다.


In [ ]:
import getpass
import http.client
import json
import re
import warnings


def check_gemini():
    model = input("GEMINI_MODEL (InfinityFree의 값): ").strip()
    if not re.fullmatch(r"[a-zA-Z0-9][a-zA-Z0-9._-]{0,99}", model):
        return {"error": "MODEL_ID_INVALID"}

    with warnings.catch_warnings():
        warnings.simplefilter("error", getpass.GetPassWarning)
        try:
            key = getpass.getpass("GEMINI_API_KEY (숨김 입력): ")
        except (getpass.GetPassWarning, EOFError, KeyboardInterrupt):
            return {"error": "HIDDEN_INPUT_UNAVAILABLE"}
    if not key or not key.isascii() or re.search(r"[\x00-\x20\x7f]", key):
        return {"error": "KEY_INPUT_INVALID"}

    prompt = "떠다니는 섬을 오가는 우편 배달부 세계관의 제목과 한 문장 소개를 한국어로 작성해 주세요."
    payload = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024},
    }
    body = json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    result = {"mode": "text", "http": 0, "valid_result": False}
    connection = http.client.HTTPSConnection("generativelanguage.googleapis.com", timeout=25)
    try:
        connection.request(
            "POST",
            f"/v1beta/models/{model}:generateContent",
            body=body,
            headers={"Content-Type": "application/json", "x-goog-api-key": key},
        )
        response = connection.getresponse()
        result["http"] = response.status
        raw = response.read(1_000_001)
    except TimeoutError:
        return {**result, "error": "GEMINI_TIMEOUT"}
    except (OSError, http.client.HTTPException):
        return {**result, "error": "GEMINI_NETWORK_ERROR"}
    finally:
        connection.close()
        del key

    try:
        decoded = json.loads(raw)
    except (ValueError, UnicodeError):
        decoded = {}
    decoded = decoded if isinstance(decoded, dict) else {}
    status = result["http"]
    if status != 200:
        detail = decoded.get("error")
        allowed = {
            "UNAVAILABLE", "RESOURCE_EXHAUSTED", "NOT_FOUND", "PERMISSION_DENIED",
            "INVALID_ARGUMENT", "UNAUTHENTICATED", "FAILED_PRECONDITION", "INTERNAL",
        }
        provider_status = detail.get("status") if isinstance(detail, dict) else None
        if isinstance(provider_status, str) and provider_status in allowed and detail.get("code") == status:
            result["provider_status"] = provider_status
        return {**result, "error": f"GEMINI_HTTP_{status}"}

    candidates = decoded.get("candidates")
    candidate = candidates[0] if isinstance(candidates, list) and candidates else {}
    candidate = candidate if isinstance(candidate, dict) else {}
    if candidate.get("finishReason") == "MAX_TOKENS":
        return {**result, "error": "GEMINI_RESPONSE_TRUNCATED"}
    content = candidate.get("content")
    parts = content.get("parts") if isinstance(content, dict) else []
    parts = parts if isinstance(parts, list) else []
    valid = candidate.get("finishReason") == "STOP" and any(
        isinstance(part, dict)
        and not part.get("thought")
        and isinstance(part.get("text"), str)
        and bool(part["text"].strip())
        for part in parts
    )
    result["valid_result"] = valid
    return result if valid else {**result, "error": "CHECK_INVALID_RESULT"}


print(json.dumps(check_gemini(), ensure_ascii=False, indent=2))
